# RF-DETR Training on SimSurgSkill Dataset

This notebook trains an RF-DETR model on the SimSurgSkill 2021 surgical dataset.

**Dataset Classes:**
- `needle` - Surgical needle
- `needle_driver` - Needle driver tool

**Requirements:**
- Google Drive access (dataset stored in Drive)
- GPU runtime recommended (T4 or better)


## 1. Setup Environment


In [ ]:
# Clone RF-DETR repository
!git clone https://github.com/NammuKall/rf-detr.git
%cd rf-detr

# Install RF-DETR with metrics dependencies
!pip install -e ".[metrics]" -q


In [ ]:
# Clone SimSurg data pipeline
!git clone https://github.com/NammuKall/simsurg_model.git /content/simsurg_model
!pip install -r /content/simsurg_model/requirements.txt -q


## 2. Mount Google Drive & Prepare Data


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Verify dataset exists
import os
GDRIVE_DATA_PATH = "/content/drive/MyDrive/Inspirit_AI_stuff/simsurgskill_2021_dataset"
assert os.path.exists(GDRIVE_DATA_PATH), f"Dataset not found at {GDRIVE_DATA_PATH}"
print(f"✅ Dataset found at: {GDRIVE_DATA_PATH}")
!ls -la "{GDRIVE_DATA_PATH}"


In [ ]:
# Create local data directory and copy dataset
!mkdir -p /content/data
!cp -r "{GDRIVE_DATA_PATH}" /content/data/simsurgskill_2021_dataset
print("✅ Dataset copied to /content/data/simsurgskill_2021_dataset")


In [ ]:
# Run SimSurg data pipeline to convert to COCO format
import sys
sys.path.insert(0, '/content/simsurg_model')

from src.data.data_extractor import VideoFrameExtractor
from src.data.data_wrangler import DataWrangler
from src.data.coco_json import COCOJSONGenerator

DATA_DIR = "/content/data/simsurgskill_2021_dataset"
COCO_OUTPUT_DIR = "/content/data/coco_format"

print("📦 Step 1: Extracting frames from videos...")
extractor = VideoFrameExtractor(fps=1)
extractor.extract_all_splits(DATA_DIR)

print("\n📦 Step 2: Wrangling annotations...")
wrangler = DataWrangler(DATA_DIR)
wrangled_data = wrangler.wrangle_all_splits()

print("\n📦 Step 3: Converting to COCO format...")
generator = COCOJSONGenerator(output_dir=COCO_OUTPUT_DIR)
coco_paths = generator.generate_from_wrangled_data(wrangled_data)

print(f"\n✅ COCO format data ready at: {COCO_OUTPUT_DIR}")


In [ ]:
# Verify COCO format structure
import json

print("📂 COCO Format Structure:")
!ls -la /content/data/coco_format/
print("\n📂 Annotations:")
!ls -la /content/data/coco_format/annotations/

# Check annotation content
with open("/content/data/coco_format/annotations/instances_train.json", "r") as f:
    train_ann = json.load(f)
    print(f"\n📊 Training Set:")
    print(f"   Images: {len(train_ann['images'])}")
    print(f"   Annotations: {len(train_ann['annotations'])}")
    print(f"   Categories: {[c['name'] for c in train_ann['categories']]}")


## 3. Setup Weights & Biases (Optional)


In [ ]:
# Install and login to W&B for experiment tracking
!pip install wandb -qU

import wandb
wandb.login()


## 4. Train RF-DETR Model


In [ ]:
# Training Configuration
CONFIG = {
    # Dataset
    "dataset_dir": "/content/data/coco_format",
    "dataset_file": "simsurg",
    
    # Training hyperparameters
    "epochs": 50,
    "batch_size": 4,
    "lr": 1e-4,
    
    # Output
    "output_dir": "/content/output",
    
    # Logging
    "wandb": True,
    "project": "simsurg-rfdetr",
    "tensorboard": True,
    
    # Early stopping
    "early_stopping": True,
    "early_stopping_patience": 10,
}

print("📋 Training Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")


In [ ]:
# Initialize and train RF-DETR model
import torch
from rfdetr import RFDETRBase

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Initialize model
print("\n🚀 Initializing RF-DETR Base model...")
model = RFDETRBase()

print("\n🎯 Starting training...")
model.train(**CONFIG)


## 5. Evaluate Results


In [ ]:
# Check training outputs
import os
import json

OUTPUT_DIR = "/content/output"

print("📂 Training Outputs:")
!ls -la {OUTPUT_DIR}/

# Load and display results
results_path = os.path.join(OUTPUT_DIR, "results.json")
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        results = json.load(f)
    print("\n📊 Training Results:")
    print(json.dumps(results, indent=2))


In [ ]:
# Display training plots
from IPython.display import Image, display
import glob

plot_files = glob.glob(f"{OUTPUT_DIR}/*.png")
for plot_file in plot_files:
    print(f"\n📊 {os.path.basename(plot_file)}")
    display(Image(filename=plot_file, width=800))


## 6. Run Inference on Test Images


In [ ]:
# Load best model and run inference
from PIL import Image as PILImage
import matplotlib.pyplot as plt
import supervision as sv
import numpy as np

# Get test images
test_images = glob.glob("/content/data/coco_format/test/images/*.jpeg")[:5]
print(f"Found {len(test_images)} test images")

# Run inference
for img_path in test_images:
    print(f"\n🔍 Processing: {os.path.basename(img_path)}")
    
    # Load image
    image = PILImage.open(img_path)
    
    # Run prediction
    detections = model.predict(image, threshold=0.5)
    
    # Annotate image
    annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    
    labels = [model.class_names.get(class_id, f"class_{class_id}") 
              for class_id in detections.class_id]
    
    annotated = annotator.annotate(
        scene=np.array(image),
        detections=detections
    )
    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections,
        labels=labels
    )
    
    # Display
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.axis('off')
    plt.title(f"{os.path.basename(img_path)} - {len(detections)} detections")
    plt.show()


## 7. Save Model to Google Drive


In [ ]:
# Save best model to Google Drive
import shutil
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/simsurg_rfdetr_models"
os.makedirs(SAVE_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_name = f"rfdetr_simsurg_{timestamp}"

# Copy checkpoint
best_checkpoint = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
if os.path.exists(best_checkpoint):
    dest_path = os.path.join(SAVE_DIR, f"{model_name}.pth")
    shutil.copy(best_checkpoint, dest_path)
    print(f"✅ Best model saved to: {dest_path}")
    
# Copy results
if os.path.exists(results_path):
    shutil.copy(results_path, os.path.join(SAVE_DIR, f"{model_name}_results.json"))
    print(f"✅ Results saved to: {SAVE_DIR}/{model_name}_results.json")


## 8. Export to ONNX (Optional)


In [ ]:
# Export model to ONNX for deployment
# Uncomment to run

# model.export(
#     output_dir="/content/onnx_export",
#     simplify=True
# )
# print("✅ ONNX model exported to /content/onnx_export")


---

## Summary

This notebook:
1. ✅ Cloned RF-DETR and SimSurg repositories
2. ✅ Mounted Google Drive and copied dataset
3. ✅ Converted SimSurg data to COCO format
4. ✅ Trained RF-DETR model with W&B logging
5. ✅ Evaluated model on test images
6. ✅ Saved model to Google Drive

**Next Steps:**
- Fine-tune hyperparameters based on results
- Try larger model variants (`RFDETRLarge`, `RFDETRMedium`)
- Export to ONNX for deployment
